In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.inchi import MolToInchi
from rdkit.Chem.Descriptors import ExactMolWt
import feos
import si_units
# 关闭RDKit的警告信息
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
def mol_to_inchi(smiles):
    mol = Chem.MolFromSmiles(smiles)
    Chem.AddHs(mol)  # 添加氢原子
    if mol is not None:
        inchi = MolToInchi(mol)
        return inchi
    else:
        return None
    
def smiles_to_moleweight(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return ExactMolWt(mol)
    else:
        return None

df_testset = pd.read_csv("TestSet.csv")
Name2SMILES = dict(zip(df_testset["Name"], df_testset["SMILES"]))
pcsaft = pd.read_csv("pcsaft_regression_results.csv")
SMILES = pcsaft["substance_name"].to_list()
INCHI = [mol_to_inchi(smiles) for smiles in SMILES]
pcsaft_m = dict(zip(INCHI, pcsaft["m"].to_list()))
pcsaft_sigma = dict(zip(INCHI, pcsaft["sigma"].to_list()))
pcsaft_epsilon_k = dict(zip(INCHI, pcsaft["epsilon_k"].to_list()))
pcsaft_mu = dict(zip(INCHI, pcsaft["mu"].to_list()))
pcsaft_kappa_ab = dict(zip(INCHI, pcsaft["kappa_ab"].to_list()))
pcsaft_epsilon_k_ab = dict(zip(INCHI, pcsaft["epsilon_k_ab"].to_list()))

gas_ = pd.read_csv("./Result/Gas_Compare.csv")
gas_Name = gas_["Name"].unique()
gas_SMILES = [Name2SMILES[name] for name in gas_Name]

liquid_ = pd.read_csv("./Result/Liquid_Compare.csv")
liquid_Name = liquid_["Name"].unique()
liquid_SMILES = [Name2SMILES[name] for name in liquid_Name]

# 根据trcid_gas_in_pcsaft构建trcid到feos.PureRecord的映射
name2pure_gas = {}
name2moleweight_gas = {}
for name, smiles in zip(gas_Name, gas_SMILES):
    inchi = mol_to_inchi(smiles)
    pure_record = feos.PureRecord(
        feos.Identifier(name=str(name)),
        molarweight=smiles_to_moleweight(smiles),
        m=pcsaft_m[inchi],
        sigma=pcsaft_sigma[inchi],
        epsilon_k=pcsaft_epsilon_k[inchi],
        mu=pcsaft_mu[inchi],
        kappa_ab=pcsaft_kappa_ab[inchi],
        epsilon_k_ab=pcsaft_epsilon_k_ab[inchi]
    )
    parameters = feos.Parameters.new_pure(pure_record)
    name2pure_gas[name] = feos.EquationOfState.pcsaft(parameters)
    name2moleweight_gas[name] = smiles_to_moleweight(smiles)
# 根据trcid_liquid_in_pcsaft构建trcid到feos.PureRecord的映射
name2pure_liquid = {}
name2moleweight_liquid = {}
for name, smiles in zip(liquid_Name, liquid_SMILES):
    inchi = mol_to_inchi(smiles)
    pure_record = feos.PureRecord(
        feos.Identifier(name=str(name)),
        molarweight=smiles_to_moleweight(smiles),
        m=pcsaft_m[inchi],
        sigma=pcsaft_sigma[inchi],
        epsilon_k=pcsaft_epsilon_k[inchi],
        mu=pcsaft_mu[inchi],
        kappa_ab=pcsaft_kappa_ab[inchi],
        epsilon_k_ab=pcsaft_epsilon_k_ab[inchi]
    )
    parameters = feos.Parameters.new_pure(pure_record)
    name2pure_liquid[name] = feos.EquationOfState.pcsaft(parameters)
    name2moleweight_liquid[name] = smiles_to_moleweight(smiles)

flag = False
for index, row in gas_.iterrows():
    name = row["Name"]
    rho = row["rho_PC-SAFT(Fit)"]
    T = row["T(K)"] * si_units.KELVIN
    P = row["P(Pa)"] * si_units.PASCAL
    eos = name2pure_gas[name]
    feos_state = feos.State(eos, temperature=T, pressure=P, density_initialization='vapor')
    rho_calculated = feos_state.density / si_units.MOL * (si_units.METER**3)
    rho_calculated = rho_calculated * name2moleweight_gas[name] / 1e+3
    if abs(rho_calculated - rho) > 1e-12:
        print(f"Gas: {name}, T: {T/si_units.KELVIN} K, P: {P/si_units.PASCAL} Pa, rho_calculated: {rho_calculated:.6f} kg/m^3, rho_PC-SAFT(Fit): {rho:.6f} kg/m^3")
        flag = True
if flag:
    print("Gas density calculation has discrepancies.")
else:
    print("All density calculations match the PC-SAFT(Fit) values.")

flag = False
for index, row in liquid_.iterrows():
    name = row["Name"]
    rho = row["rho_PC-SAFT(Fit)"]
    T = row["T(K)"] * si_units.KELVIN
    P = row["P(Pa)"] * si_units.PASCAL
    eos = name2pure_liquid[name]
    feos_state = feos.State(eos, temperature=T, pressure=P, density_initialization='liquid')
    rho_calculated = feos_state.density / si_units.MOL * (si_units.METER**3)
    rho_calculated = rho_calculated * name2moleweight_liquid[name] / 1e+3
    if abs(rho_calculated - rho) > 1e-12:
        print(f"Liquid: {name}, T: {T/si_units.KELVIN} K, P: {P/si_units.PASCAL} Pa, rho_calculated: {rho_calculated:.6f} kg/m^3, rho_PC-SAFT(Fit): {rho:.6f} kg/m^3")
        flag = True


if flag:
    print("Liquid density calculation has discrepancies.")
else:
    print("All density calculations match the PC-SAFT(Fit) values.")

All density calculations match the PC-SAFT(Fit) values.
All density calculations match the PC-SAFT(Fit) values.
